# Baltic exploration — starter notebook

Loads Danish AIS for the Eagle S incident day (Dec 25, 2024), the criticality layers, and produces a folium map showing vessel tracks over Baltic infrastructure.

**Prerequisites:** AWS credentials configured (`aws configure`), Python deps installed (`pip install -r requirements.txt`).

## Sections
1. AIS load + sanity
2. Criticality layers
3. Eagle S vessel track (MMSI 273347410) overlaid on Estlink 2
4. Suspicious-behavior heuristics: speed-near-cable, AIS dropouts

## 1 · AIS load + sanity

In [ ]:
import os
import configparser
import duckdb
import geopandas as gpd
import pandas as pd
import folium
from pathlib import Path

# Wire AWS credentials into DuckDB
cp = configparser.ConfigParser()
cp.read(os.path.expanduser('~/.aws/credentials'))
AK = cp['default']['aws_access_key_id']
SK = cp['default']['aws_secret_access_key']
con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute(f"CREATE SECRET (TYPE s3, KEY_ID '{AK}', SECRET '{SK}', REGION 'eu-west-3')")

In [ ]:
# Load Eagle S day from S3 via DuckDB (zero-copy, no local download)
S3_PARQUET = "s3://edth2026-baltic/ais/parquet/source=danish/year=2024/month=12/day=25/part-0000.parquet"

df = con.execute(f"""
    SELECT MMSI, Name, \"Ship type\", Timestamp, Latitude, Longitude, SOG, COG, Heading, IMO, Callsign, ts
    FROM read_parquet('{S3_PARQUET}')
    WHERE Latitude BETWEEN 52 AND 66 AND Longitude BETWEEN 9 AND 30
""").df()
print(f'rows: {len(df):,}')
print(f'unique vessels: {df["MMSI"].nunique():,}')
print(f'time range: {df["ts"].min()} → {df["ts"].max()}')
df.head(3)

## 2 · Criticality layers

In [ ]:
# Auto-pull large layers from S3 if missing locally
GEO = Path('../../geo')   # relative to data/samples/notebooks/
for name in ('submarine_cables', 'submarine_power_cables', 'pipelines', 'naval_bases',
             'ports', 'refineries_lng', 'offshore_wind', 'offshore_platforms',
             'emodnet_pipelines', 'emodnet_windfarmspoly',
             'emodnet_cables_combined', 'marine_regions_eez_baltic'):
    path = GEO / f'{name}.geojson'
    if not path.exists():
        os.system(f'aws s3 cp s3://edth2026-baltic/geo/{name}.geojson {path}')

cables = gpd.read_file(GEO / 'submarine_cables.geojson')
power_cables = gpd.read_file(GEO / 'submarine_power_cables.geojson')
pipelines = gpd.read_file(GEO / 'pipelines.geojson')
naval = gpd.read_file(GEO / 'naval_bases.geojson')
print(f'cables: {len(cables)}, power_cables: {len(power_cables)}, pipelines: {len(pipelines)}, naval: {len(naval)}')

## 3 · Eagle S vessel track (MMSI 273347410) overlaid on Estlink 2

In [ ]:
# Eagle S — Cook Islands tanker that severed Estlink 2 on Christmas Eve 2024
EAGLE_S_MMSI = 273347410

eagle = df[df['MMSI'] == EAGLE_S_MMSI].sort_values('ts')
print(f'Eagle S points in Danish AIS: {len(eagle)}')
if len(eagle) > 0:
    print(f'Name: {eagle["Name"].dropna().unique()}')
    print(f'Ship type: {eagle["Ship type"].dropna().unique()}')
    print(f'Bounding box: lat {eagle["Latitude"].min():.4f}-{eagle["Latitude"].max():.4f}, lon {eagle["Longitude"].min():.4f}-{eagle["Longitude"].max():.4f}')
    print(f'Speed range (SOG): {eagle["SOG"].min():.2f} - {eagle["SOG"].max():.2f} kn')
eagle.head(5)

In [ ]:
# Folium map centered on Gulf of Finland
m = folium.Map(location=[60.0, 26.0], zoom_start=8, tiles='CartoDB positron')

# Cables layer
folium.GeoJson(
    cables.to_json(),
    name='Submarine cables',
    style_function=lambda f: {'color': '#888', 'weight': 1}
).add_to(m)
folium.GeoJson(
    power_cables.to_json(),
    name='Power cables',
    style_function=lambda f: {'color': '#cc6600', 'weight': 2}
).add_to(m)
folium.GeoJson(
    pipelines[pipelines.geometry.type == 'LineString'].to_json(),
    name='Pipelines',
    style_function=lambda f: {'color': '#990033', 'weight': 1.5, 'opacity': 0.7}
).add_to(m)

# Eagle S track
if len(eagle) > 0:
    points = list(zip(eagle['Latitude'], eagle['Longitude']))
    folium.PolyLine(points, color='red', weight=4, opacity=0.9, tooltip='Eagle S track').add_to(m)
    # First and last position markers
    folium.CircleMarker(points[0], radius=6, color='green',
                         tooltip=f'Eagle S start: {eagle["ts"].iloc[0]}').add_to(m)
    folium.CircleMarker(points[-1], radius=6, color='black',
                         tooltip=f'Eagle S end: {eagle["ts"].iloc[-1]}').add_to(m)

folium.LayerControl().add_to(m)
m

## 4 · Naive suspicion heuristic: slow vessels over cables

In [ ]:
# Project both AIS points and cables to a local UTM CRS for distance math
from shapely.geometry import Point

# Convert dataframe slice to GeoDataFrame (just a manageable sample)
sample = df.sample(n=min(100000, len(df)), random_state=42).copy()
gdf = gpd.GeoDataFrame(
    sample,
    geometry=[Point(xy) for xy in zip(sample['Longitude'], sample['Latitude'])],
    crs='EPSG:4326',
)
# Project to UTM zone 35N (covers Gulf of Finland)
gdf_m = gdf.to_crs(epsg=32635)
cables_m = power_cables.to_crs(epsg=32635)

# Distance from each AIS point to nearest power cable (km)
from shapely.ops import unary_union
cable_union = unary_union(cables_m.geometry.values)
gdf_m['dist_cable_km'] = gdf_m.distance(cable_union) / 1000.0

# Suspicious = slow (SOG < 4 kn) AND close to a cable (<2 km) AND not in a port
suspicious = gdf_m[(gdf_m['SOG'] < 4) & (gdf_m['SOG'] > 0.1) & (gdf_m['dist_cable_km'] < 2.0)].copy()
print(f'Suspicious points (slow near cable): {len(suspicious)} of {len(gdf_m)}')
print(f'Distinct MMSIs flagged: {suspicious["MMSI"].nunique()}')
suspicious_summary = suspicious.groupby('MMSI').agg(
    name=('Name', lambda x: x.dropna().iloc[0] if x.dropna().size else ''),
    ship_type=('Ship type', lambda x: x.dropna().iloc[0] if x.dropna().size else ''),
    n_points=('SOG', 'size'),
    min_sog=('SOG', 'min'),
    min_dist_km=('dist_cable_km', 'min')
).sort_values('n_points', ascending=False).head(20)
suspicious_summary

## What to build next

- **Class-conditional behavior model** (§5.1 of PLAN.md): per-vessel-type trajectory models. Score the log-likelihood of a vessel's track under its *declared* class.
- **Criticality surface rasterization** (§5.2): grid the Baltic at ~1 km, sum proximity-weighted scores from each criticality layer.
- **Tip-and-cue output** (§5.5): aggregate suspicion into geographic boxes, rank, output "task next satellite pass" recommendations.